## Step 1: The TESS ID Intake Form

Before we can download anything from TESS, we need a list of TIC (TESS Input Catalog) IDs to actually go fetch. This notebook is a minimal, no-frills "type in some numbers" collector — the TESS equivalent of the Kepler filename collector from the first pipeline, just with less typing since TIC IDs are plain integers instead of essay-length filenames.

**Why this exists:** Rather than hardcoding a list of star IDs into every script, we type them once here and save them to a CSV that later notebooks (`id_check`, `dataset_2`) can read from — a single source of truth for "which stars are we investigating."

**The Algorithm:**
1. **Initialize an empty list** (`lis = []`) and a counter (`i = 0`) to track how many IDs have been entered.
2. **Loop indefinitely** using a sentinel flag `c = 1`, prompting `Enter the {i+1}th id:` each time.
3. **Try to parse the input as an integer** — if it succeeds, append it to the list and increment the counter. If it fails (i.e. the user types something that isn't a number, like `q` to quit), a `ValueError` is caught, `"EXITING PROGRAM"` is printed, and the loop flag is flipped to `0`, ending collection. This is a clever (if slightly accidental) use of the `try/except` block as an exit mechanism — type any non-numeric string to quit gracefully.
4. **Deduplicate the list** using `list(dict.fromkeys(lis))` — a neat Python idiom that removes duplicates while preserving insertion order (unlike `set()`, which would scramble the order).
5. **Wrap the deduplicated IDs into a DataFrame** with a single column, `tess_id`.
6. **Append to CSV** (`new_tess_id.csv`, `mode='a'`) — same pattern as the Kepler notebook, bolting new entries onto the existing file rather than overwriting it.
7. **Read it back** (`dc = pd.read_csv(...)`) as a final sanity check that the write actually worked.

**Output:** `new_tess_id.csv` — a single-column CSV of `tess_id` values, consumed by `02_id_check.ipynb` next.

**Gotcha to know:** Just like the Kepler version, this dedups only *within* the current run before appending — it doesn't check against IDs already saved in the CSV from previous runs. Running this notebook twice with overlapping IDs will create duplicate rows in the file itself, so a periodic `drop_duplicates()` pass on the saved CSV is a good habit.

*In short: this notebook is a glorified `input()` loop with commitment issues about when to stop — solved by "type garbage to exit."*

In [1]:
import pandas as pd
import numpy as np

In [2]:
lis = []

In [3]:
i = 0

In [4]:
c = 1

In [5]:
file_name = 'new_tess_id.csv'

In [6]:
while c !=0:
    try:
        id = int(input(f'Enter the {i+1}th id: '))
        lis.append(id)
        i+=1
    except ValueError:
        print('EXITING PROGRAM')
        c = 0

Enter the 1th id:  100100827
Enter the 2th id:  92352620
Enter the 3th id:  149603524
Enter the 4th id:  25155310
Enter the 5th id:  281541555
Enter the 6th id:  261136679
Enter the 7th id:  q


EXITING PROGRAM


In [7]:
len(lis)

6

In [8]:
li = list(dict.fromkeys(lis))

In [9]:
len(li)

6

In [10]:
di = {"tess_id" : li}

In [11]:
df = pd.DataFrame(di)

In [12]:
df.to_csv(file_name, mode='a', index=False)

In [13]:
dc = pd.read_csv('new_tess_id.csv')